In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, trim, lower, when, lit, current_timestamp,
    to_json, struct
)
import uuid

# =========================================================
# CONFIG
# =========================================================
pipeline_name = "retail_pipeline"
layer_name = "silver"
load_type = "FULL"
triggered_by = "manual"
source_system = "Bronze_Tables"

audit_table = "retail.audit.etl_run_log"
reject_table = "retail.audit.etl_reject_log"

batch_id = "batch_silver_full_20260423"

table_configs = [
    {
        "table_name": "ns_brands",
        "source_table": "retail.bronze.ns_brands",
        "target_table": "retail.silver.ns_brands",
        "business_key": "internal_id",
        "typed_columns": [
            ("internal_id", "int"),
            ("brand_code", "string"),
            ("brand_name", "string"),
            ("is_active", "boolean_flag"),
            ("created_at", "timestamp"),
            ("etl_loaded_at", "timestamp"),
            ("source_system", "string"),
            ("etl_run_id", "string"),
            ("ingestion_date", "date"),
            ("record_hash", "string"),
            ("is_deleted", "boolean_flag")
        ],
        "required_columns": ["internal_id", "brand_code", "brand_name", "created_at"],
        "dedupe_keys": ["internal_id", "record_hash"]
    },
    {
        "table_name": "ns_customers",
        "source_table": "retail.bronze.ns_customers",
        "target_table": "retail.silver.ns_customers",
        "business_key": "internal_id",
        "typed_columns": [
            ("internal_id", "int"),
            ("entity_id", "string"),
            ("company_name", "string"),
            ("customer_type", "string"),
            ("email", "string"),
            ("phone", "string"),
            ("subsidiary", "string"),
            ("currency_code", "string"),
            ("terms_name", "string"),
            ("vat_number", "string"),
            ("billing_address1", "string"),
            ("billing_city", "string"),
            ("billing_state", "string"),
            ("billing_country", "string"),
            ("shipping_address1", "string"),
            ("shipping_city", "string"),
            ("shipping_state", "string"),
            ("shipping_country", "string"),
            ("is_active", "boolean_flag"),
            ("created_at", "timestamp"),
            ("etl_loaded_at", "timestamp"),
            ("source_system", "string"),
            ("etl_run_id", "string"),
            ("ingestion_date", "date"),
            ("record_hash", "string"),
            ("is_deleted", "boolean_flag")
        ],
        "required_columns": [
            "internal_id", "entity_id", "company_name", "customer_type",
            "subsidiary", "currency_code", "terms_name", "created_at"
        ],
        "dedupe_keys": ["internal_id", "record_hash"]
    },
    {
        "table_name": "ns_items",
        "source_table": "retail.bronze.ns_items",
        "target_table": "retail.silver.ns_items",
        "business_key": "internal_id",
        "typed_columns": [
            ("internal_id", "int"),
            ("item_id", "string"),
            ("item_name", "string"),
            ("item_type", "string"),
            ("category", "string"),
            ("subcategory", "string"),
            ("brand_internal_id", "int_nullable"),
            ("base_price", "decimal"),
            ("cost_price", "decimal"),
            ("is_active", "boolean_flag"),
            ("created_at", "timestamp"),
            ("etl_loaded_at", "timestamp"),
            ("source_system", "string"),
            ("etl_run_id", "string"),
            ("ingestion_date", "date"),
            ("record_hash", "string"),
            ("is_deleted", "boolean_flag")
        ],
        "required_columns": [
            "internal_id", "item_id", "item_name", "item_type",
            "category", "subcategory", "base_price", "cost_price", "created_at"
        ],
        "dedupe_keys": ["internal_id", "record_hash"]
    },
    {
        "table_name": "ns_sales_orders",
        "source_table": "retail.bronze.ns_sales_orders",
        "target_table": "retail.silver.ns_sales_orders",
        "business_key": "internal_id",
        "typed_columns": [
            ("internal_id", "int"),
            ("tran_id", "string"),
            ("customer_internal_id", "int"),
            ("order_date", "date"),
            ("order_status", "string"),
            ("sales_channel", "string"),
            ("payment_status", "string"),
            ("currency_code", "string"),
            ("external_ref_num", "string"),
            ("integration_source", "string"),
            ("order_total", "decimal"),
            ("created_at", "timestamp"),
            ("etl_loaded_at", "timestamp"),
            ("source_system", "string"),
            ("etl_run_id", "string"),
            ("ingestion_date", "date"),
            ("record_hash", "string"),
            ("is_deleted", "boolean_flag")
        ],
        "required_columns": [
            "internal_id", "tran_id", "customer_internal_id", "order_date",
            "order_status", "sales_channel", "payment_status", "currency_code",
            "integration_source", "order_total", "created_at"
        ],
        "dedupe_keys": ["internal_id", "record_hash"]
    },
    {
        "table_name": "ns_sales_order_lines",
        "source_table": "retail.bronze.ns_sales_order_lines",
        "target_table": "retail.silver.ns_sales_order_lines",
        "business_key": "internal_id",
        "typed_columns": [
            ("internal_id", "int"),
            ("sales_order_internal_id", "int"),
            ("line_num", "int"),
            ("item_internal_id", "int"),
            ("quantity", "int"),
            ("unit_rate", "decimal"),
            ("line_amount", "decimal"),
            ("created_at", "timestamp"),
            ("etl_loaded_at", "timestamp"),
            ("source_system", "string"),
            ("etl_run_id", "string"),
            ("ingestion_date", "date"),
            ("record_hash", "string"),
            ("is_deleted", "boolean_flag")
        ],
        "required_columns": [
            "internal_id", "sales_order_internal_id", "line_num",
            "item_internal_id", "quantity", "unit_rate", "line_amount", "created_at"
        ],
        "dedupe_keys": ["internal_id", "record_hash"]
    },
    {
        "table_name": "ns_shipments",
        "source_table": "retail.bronze.ns_shipments",
        "target_table": "retail.silver.ns_shipments",
        "business_key": "internal_id",
        "typed_columns": [
            ("internal_id", "int"),
            ("shipment_number", "string"),
            ("sales_order_internal_id", "int"),
            ("warehouse_name", "string"),
            ("shipment_date", "date"),
            ("delivery_date", "date_nullable"),
            ("shipment_status", "string"),
            ("tracking_number", "string"),
            ("created_at", "timestamp"),
            ("etl_loaded_at", "timestamp"),
            ("source_system", "string"),
            ("etl_run_id", "string"),
            ("ingestion_date", "date"),
            ("record_hash", "string"),
            ("is_deleted", "boolean_flag")
        ],
        "required_columns": [
            "internal_id", "shipment_number", "sales_order_internal_id",
            "warehouse_name", "shipment_date", "shipment_status", "created_at"
        ],
        "dedupe_keys": ["internal_id", "record_hash"]
    }
]

# =========================================================
# HELPERS
# =========================================================
def log_layer_processed_runs_full(table_name, target_run_id):
    bronze_runs = spark.sql(f"""
        SELECT DISTINCT run_id
        FROM retail.audit.ingested_files
        WHERE table_name = '{table_name}'
          AND ingestion_status = 'SUCCESS'
    """)

    bronze_runs_list = [r.run_id for r in bronze_runs.collect()]

    for bronze_run_id in bronze_runs_list:
        spark.sql(f"""
            INSERT INTO retail.audit.layer_processed_runs
            SELECT
              uuid() AS tracking_id,
              '{bronze_run_id}' AS source_run_id,
              '{target_run_id}' AS target_run_id,
              '{pipeline_name}' AS pipeline_name,
              'bronze' AS source_layer,
              'silver' AS target_layer,
              '{table_name}' AS table_name,
              'SUCCESS' AS status,
              NULL AS rows_read,
              NULL AS rows_written,
              NULL AS rows_rejected,
              current_timestamp() AS start_time,
              current_timestamp() AS end_time,
              0 AS duration_seconds,
              NULL AS error_message,
              current_timestamp() AS created_at,
              current_timestamp() AS updated_at
        """)

def generate_run_id():
    return str(uuid.uuid4())

def start_audit(run_id, table_name):
    spark.sql(f"""
        INSERT INTO {audit_table}
        (
          run_id, batch_id, pipeline_name, layer_name, table_name, source_system,
          load_type, triggered_by, start_time, end_time, duration_seconds,
          status, rows_read, rows_written, rows_rejected, records_before,
          records_after, error_message, created_at
        )
        VALUES (
          '{run_id}', '{batch_id}', '{pipeline_name}', '{layer_name}', '{table_name}', '{source_system}',
          '{load_type}', '{triggered_by}', current_timestamp(), NULL, NULL,
          'STARTED', NULL, NULL, NULL, NULL,
          NULL, NULL, current_timestamp()
        )
    """)

def end_audit_success(run_id, rows_read, rows_written, rows_rejected, records_before, records_after):
    spark.sql(f"""
        UPDATE {audit_table}
        SET
          end_time = current_timestamp(),
          duration_seconds = unix_timestamp(current_timestamp()) - unix_timestamp(start_time),
          status = 'SUCCESS',
          rows_read = {rows_read},
          rows_written = {rows_written},
          rows_rejected = {rows_rejected},
          records_before = {records_before},
          records_after = {records_after}
        WHERE run_id = '{run_id}'
    """)

def end_audit_fail(run_id, error_message):
    safe_error = error_message.replace("'", "''")[:5000]
    spark.sql(f"""
        UPDATE {audit_table}
        SET
          end_time = current_timestamp(),
          duration_seconds = unix_timestamp(current_timestamp()) - unix_timestamp(start_time),
          status = 'FAILED',
          error_message = '{safe_error}'
        WHERE run_id = '{run_id}'
    """)

def log_rejects(df_rejects, run_id, table_name):
    if df_rejects.isEmpty():
        return

    (
        df_rejects
        .withColumn("reject_id", F.expr("uuid()"))
        .withColumn("run_id", lit(run_id))
        .withColumn("batch_id", lit(batch_id))
        .withColumn("pipeline_name", lit(pipeline_name))
        .withColumn("layer_name", lit(layer_name))
        .withColumn("table_name", lit(table_name))
        .withColumn("source_system", lit(source_system))
        .withColumn("rejected_at", current_timestamp())
        .withColumn("created_at", current_timestamp())
        .select(
            "reject_id", "run_id", "batch_id", "pipeline_name", "layer_name",
            "table_name", "source_system", "source_record_id", "reject_stage",
            "reject_reason", "reject_column", "severity", "raw_payload",
            "error_code", "error_message", "rejected_at", "created_at"
        )
        .write.format("delta").mode("append").saveAsTable(reject_table)
    )

def normalize_string(c):
    return when(trim(col(c)) == "", None).otherwise(trim(col(c)))

def build_typed_expr(source_col, rule):
    raw = normalize_string(source_col)

    if rule == "string":
        return raw

    if rule == "int":
        return raw.cast("int")

    if rule == "int_nullable":
        return raw.cast("int")

    if rule == "decimal":
        return raw.cast("decimal(18,2)")

    if rule == "timestamp":
        return raw.cast("timestamp")

    if rule == "date":
        return raw.cast("date")

    if rule == "date_nullable":
        return raw.cast("date")

    if rule == "boolean_flag":
        return (
            when(lower(raw).isin("true", "1", "yes"), F.lit(True))
            .when(lower(raw).isin("false", "0", "no"), F.lit(False))
            .otherwise(F.lit(False))
        )

    return raw

def build_invalid_condition(df, typed_columns, required_columns):
    condition = F.lit(False)

    for c in required_columns:
        condition = condition | col(c).isNull() | (trim(col(c)) == "")

    for c, rule in typed_columns:
        if rule == "int":
            condition = condition | (
                normalize_string(c).isNotNull() &
                build_typed_expr(c, "int").isNull()
            )
        elif rule == "decimal":
            condition = condition | (
                normalize_string(c).isNotNull() &
                build_typed_expr(c, "decimal").isNull()
            )
        elif rule == "timestamp":
            condition = condition | (
                normalize_string(c).isNotNull() &
                build_typed_expr(c, "timestamp").isNull()
            )
        elif rule == "date":
            condition = condition | (
                normalize_string(c).isNotNull() &
                build_typed_expr(c, "date").isNull()
            )
        elif rule == "boolean_flag":
            condition = condition | (
                normalize_string(c).isNotNull() &
                (~lower(normalize_string(c)).isin("true", "1", "yes", "false", "0", "no"))
            )

    return condition

# =========================================================
# MAIN TABLE PROCESSOR
# =========================================================
def process_table(cfg):
    table_name = cfg["table_name"]
    source_table = cfg["source_table"]
    target_table = cfg["target_table"]
    business_key = cfg["business_key"]
    typed_columns = cfg["typed_columns"]
    required_columns = cfg["required_columns"]
    dedupe_keys = cfg["dedupe_keys"]

    run_id = generate_run_id()
    start_audit(run_id, table_name)

    try:
        df_bronze = spark.table(source_table)
        rows_read = df_bronze.count()

        records_before = spark.table(target_table).count() if spark.catalog.tableExists(target_table) else 0

        invalid_condition = build_invalid_condition(df_bronze, typed_columns, required_columns)

        df_rejects = (
            df_bronze
            .filter(invalid_condition)
            .withColumn("source_record_id", normalize_string(business_key))
            .withColumn("reject_stage", lit("SILVER"))
            .withColumn("reject_reason", lit("INVALID_CAST_OR_REQUIRED_FIELD"))
            .withColumn("reject_column", lit(",".join(required_columns)))
            .withColumn("severity", lit("ERROR"))
            .withColumn("raw_payload", to_json(struct(*[col(c) for c in df_bronze.columns])))
            .withColumn("error_code", lit("SLV_001"))
            .withColumn("error_message", lit("Required field missing/blank or failed type cast"))
        )

        rows_rejected = df_rejects.count()
        log_rejects(df_rejects, run_id, table_name)

        df_valid = df_bronze.filter(~invalid_condition)

        typed_select_exprs = [
            build_typed_expr(c, rule).alias(c)
            for c, rule in typed_columns
        ]

        df_silver = (
            df_valid
            .select(*typed_select_exprs)
            .dropDuplicates(dedupe_keys)
        )

        target_cols = spark.table(target_table).columns
        df_silver = df_silver.select(*target_cols)

        rows_written = df_silver.count()

        df_silver.write \
            .mode("overwrite") \
            .format("delta") \
            .option("overwriteSchema", "true") \
            .saveAsTable(target_table)

        records_after = spark.table(target_table).count()

        end_audit_success(
            run_id=run_id,
            rows_read=rows_read,
            rows_written=rows_written,
            rows_rejected=rows_rejected,
            records_before=records_before,
            records_after=records_after
        )

        log_layer_processed_runs_full(
            table_name=table_name,
            target_run_id=run_id
        )

        print(f"{target_table}: {rows_written} rows loaded successfully")

    except Exception as e:
        end_audit_fail(run_id, str(e))
        raise

# =========================================================
# RUN ALL TABLES
# =========================================================
for cfg in table_configs:
    process_table(cfg)